## Step 1: Preprocess

In [ ]:
# Step 1: Loading Data and Basic Preprocessing

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset
file_path = "/content/combined_data_1.txt"  # Adjust filename if needed
df = pd.read_csv(file_path, header=None, names=['Cust_Id', 'Rating'], usecols=[0,1])

# Check the shape of the data and other initial details
print(df.head())
df.info()

# Count the number of ratings given by customers
ratings_count = pd.DataFrame(df['Rating'].value_counts().sort_index(ascending=True))
ratings_count

# Calculate movie and customer counts
movie_count = df.isnull().sum()[1]
customer_count = df['Cust_Id'].nunique() - movie_count

# Aggregate rating count
Total_ratings_count = ratings_count.sum()[0]

# Visualize ratings distribution
ax = ratings_count.plot(kind='barh', legend=False, figsize=(15,10))
plt.title(f"Total Pool: {movie_count} Movies, {customer_count} Customers, {Total_ratings_count} Ratings given", fontsize=10)
plt.xlabel("Count Of Rating")
plt.ylabel("Rating")
plt.show()

# Handling missing values and setting up Movie IDs
df_nan = pd.DataFrame(df['Rating'].isnull())
df_nan = df_nan[df_nan['Rating'] == True]

# Reset index and create movie IDs
df_nan.reset_index(inplace=True)
movie_np = []

movie_id = 1
for i, j in zip(df_nan['index'][1:], df_nan['index'][:-1]):
    temp = np.full((1, i-j-1), movie_id)
    movie_np = np.append(movie_np, temp)
    movie_id += 1

last_record = np.full((1, len(df) - df_nan.iloc[-1,0]-1), movie_id)
movie_np = np.append(movie_np, last_record)

# Remove NaN and add 'Movie_Id' column
df = df[pd.notnull(df['Rating'])]
df['Movie_Id'] = movie_np.astype(int)
df['Cust_Id'] = df['Cust_Id'].astype(int)


## Step 2: SVD

In [ ]:
# Step 2: Applying SVD

from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate, train_test_split

# Prepare the dataset for SVD
reader = Reader()
surprise_data = Dataset.load_from_df(df[['Cust_Id', 'Movie_Id', 'Rating']], reader)

# Initialize and train the SVD model
svd = SVD()
cross_validate(svd, surprise_data, measures=['RMSE', 'MAE'], cv=2, verbose=True)


## Recommendations

In [ ]:
# Step 3: Generating Recommendations and Evaluation

# Get top movie recommendations per genre for a user
def get_top_movies_per_genre(user_id, data, algo):
    genres = data['Genres'].unique()
    top_movies = []
    rated_movie_ids = set(data[data['Cust_Id'] == user_id]['Movie_Id'])
    avg_ratings = data.groupby('Movie_Id')['Rating'].mean().to_dict()

    for genre in genres:
        genre_movie = data[data['Genres'] == genre]['Movie_Id'].unique()
        unrated_genre_movies = [movie_id for movie_id in genre_movie if movie_id not in rated_movie_ids]
        predictions = [algo.predict(user_id, movie_id) for movie_id in unrated_genre_movies]
        predictions.sort(key=lambda x: x.est, reverse=True)

        if predictions:
            top_movie = predictions[0]
            movie_title = data[data['Movie_Id'] == top_movie.iid]['Title'].iloc[0]
            avg_rating = avg_ratings.get(top_movie.iid, "No Rating")
            top_movies.append({'Movie_Id': top_movie.iid,
                               'Title': movie_title,
                               'Rating': avg_rating,
                               'Genre': genre,
                               'Estimated Score': top_movie.est})

    return pd.DataFrame(top_movies)

# Predict for a random user
import random
user_ids = df['Cust_Id'].unique()
User_Id = random.choice(user_ids)

top_movies_per_genre = get_top_movies_per_genre(User_Id, df, svd)
top_movies_per_genre = top_movies_per_genre.sort_values(by='Estimated Score', ascending=False).reset_index(drop=True)
print(f"\033[1m Best Suited Movie For {User_Id} In Every Genre: ")
top_movies_per_genre.head(50)
